In [13]:
import torch
import torchaudio
import speechbrain

print("Torch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("SpeechBrain:", speechbrain.__version__)


Torch: 2.8.0
Torchaudio: 2.8.0
SpeechBrain: 1.0.3


In [14]:
from pathlib import Path

DATA_ROOT = Path("../data/voxceleb1/test/wav")
OUT_DIR = Path("../outputs/xvectors")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Speakers:", len(list(DATA_ROOT.iterdir())))


Speakers: 41


In [15]:
from pathlib import Path

DATA_ROOT = Path("../data/voxceleb1/test/wav")

print("DATA_ROOT exists:", DATA_ROOT.exists())

# Count wav files recursively
wav_files = list(DATA_ROOT.rglob("*.wav"))
print("Total wav files found:", len(wav_files))

# Show one example path (very important)
if wav_files:
    print("Example wav:", wav_files[0])


DATA_ROOT exists: True
Total wav files found: 4874
Example wav: ../data/voxceleb1/test/wav/id10295/nt7dNRvlEHE/00005.wav


In [16]:
from speechbrain.inference import EncoderClassifier

classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-xvect-voxceleb",
    run_opts={"device": "cpu"}  # change to "cuda" if GPU
)


In [17]:
import numpy as np
xvectors = []
utt_ids = []
speaker_ids = []

for wav_path in DATA_ROOT.rglob("*.wav"):
    spk_id = wav_path.parts[-3]   # id10295
    utt_id = wav_path.stem        # 00005

    signal, sr = torchaudio.load(wav_path)
    emb = classifier.encode_batch(signal)
    emb = emb.squeeze().detach().cpu().numpy()

    xvectors.append(emb)
    # Relative path used in protocol
    rel = wav_path.relative_to(DATA_ROOT)
    utt_ids.append(str(rel))
    speaker_ids.append(rel.parts[0])  # id10270

xvectors = np.stack(xvectors)

print("X-vectors shape:", xvectors.shape)


/opt/anaconda3/envs/speechbrain/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


X-vectors shape: (4874, 512)


In [18]:
np.save(OUT_DIR / "xvectors.npy", xvectors)
np.save(OUT_DIR / "utterances.npy", np.array(utt_ids))
np.save(OUT_DIR / "speakers.npy", np.array(speaker_ids))

In [19]:
import pandas as pd 
df = pd.DataFrame({
    "utterance": utt_ids,
    "speaker": speaker_ids
})

df.to_csv(OUT_DIR / "metadata.csv", index=False)


In [12]:
import urllib.request
from pathlib import Path

# Where your test data lives
DATA_ROOT = Path("../data/voxceleb1/test")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

veri_url = "https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt"
veri_path = DATA_ROOT / "veri_test2.txt"

if not veri_path.exists():
    print("Downloading veri_test2.txt ...")
    urllib.request.urlretrieve(veri_url, veri_path)
    print("Downloaded to:", veri_path)
else:
    print("veri_test2.txt already exists at:", veri_path)


Downloaded to: ../data/voxceleb1/test/veri_test2.txt
